In [1]:
import duckdb
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder

con=duckdb.connect('ercot_weather.duckdb')

df_model = con.sql('''
    SELECT *
    FROM demand_weather_model
    ORDER BY period_utc
''').df()

con.close()

df_model.head()

,period_utc,value,hour,dayofweek,month,year,lag_1,lag_24,lag_168,rolling_mean_24,...,extreme_demand_flag,temp_mean,temp_max,temp_min,humid_mean,dewpoint_mean,cloud_cover_mean,apparent_temp_mean,wind_speed_mean,precip_mean
0,2019-01-01 00:00:00-06:00,37301.0,6,1,1,2019,NaN,NaN,NaN,NaN,...,False,44.088047,54.128300,37.580002,83.323517,39.213051,20.166666,39.003639,6.370968,0.0
1,2019-01-01 01:00:00-06:00,36953.0,7,1,1,2019,37301.0,NaN,NaN,NaN,...,False,42.678051,55.478302,34.340000,87.261353,39.078049,31.833334,37.659748,6.213417,0.0
2,2019-01-01 02:00:00-06:00,37114.0,8,1,1,2019,36953.0,NaN,NaN,NaN,...,False,41.748051,56.918301,31.280001,90.389404,39.093052,28.500000,36.578342,6.587480,0.0
3,2019-01-01 03:00:00-06:00,37154.0,9,1,1,2019,37114.0,NaN,NaN,NaN,...,False,40.833050,50.888298,28.940001,88.685234,37.683048,27.833334,34.419422,8.797850,0.0
4,2019-01-01 04:00:00-06:00,37290.0,10,1,1,2019,37154.0,NaN,NaN,NaN,...,False,40.158051,52.868301,27.410000,86.765106,36.423050,11.500000,33.537193,8.912553,0.0


In [2]:
df_model.shape

(61345, 22)

Feature engineering for increased demand ranges.

In [3]:
df_model['is_weekend'] = df_model['dayofweek'].isin([5,6])
df_model['is_summer'] = df_model['month'].isin([6,7,8,9])
df_model['is_winter'] = df_model['month'].isin([12,1,2])

These are the historical dates that have lead to increased demand. May increase model accuracy when predicting.

In [4]:
df_model.head()

,period_utc,value,hour,dayofweek,month,year,lag_1,lag_24,lag_168,rolling_mean_24,...,temp_min,humid_mean,dewpoint_mean,cloud_cover_mean,apparent_temp_mean,wind_speed_mean,precip_mean,is_weekend,is_summer,is_winter
0,2019-01-01 00:00:00-06:00,37301.0,6,1,1,2019,NaN,NaN,NaN,NaN,...,37.580002,83.323517,39.213051,20.166666,39.003639,6.370968,0.0,False,False,True
1,2019-01-01 01:00:00-06:00,36953.0,7,1,1,2019,37301.0,NaN,NaN,NaN,...,34.340000,87.261353,39.078049,31.833334,37.659748,6.213417,0.0,False,False,True
2,2019-01-01 02:00:00-06:00,37114.0,8,1,1,2019,36953.0,NaN,NaN,NaN,...,31.280001,90.389404,39.093052,28.500000,36.578342,6.587480,0.0,False,False,True
3,2019-01-01 03:00:00-06:00,37154.0,9,1,1,2019,37114.0,NaN,NaN,NaN,...,28.940001,88.685234,37.683048,27.833334,34.419422,8.797850,0.0,False,False,True
4,2019-01-01 04:00:00-06:00,37290.0,10,1,1,2019,37154.0,NaN,NaN,NaN,...,27.410000,86.765106,36.423050,11.500000,33.537193,8.912553,0.0,False,False,True


In [5]:
df_model.groupby('is_weekend')['dayofweek'].unique()

is_weekend
False    [1, 2, 3, 4, 0]
True              [5, 6]
Name: dayofweek, dtype: object

In [6]:
df_model.groupby('is_summer')['month'].unique()

is_summer
False    [1, 2, 3, 4, 5, 10, 11, 12]
True                    [6, 7, 8, 9]
Name: month, dtype: object

In [7]:
df_model.groupby('is_winter')['month'].unique()

is_winter
False    [3, 4, 5, 6, 7, 8, 9, 10, 11]
True                        [1, 2, 12]
Name: month, dtype: object

In [8]:
df_model.isna().sum()

period_utc               0
value                    0
hour                     0
dayofweek                0
month                    0
year                     0
lag_1                    1
lag_24                  24
lag_168                168
rolling_mean_24         24
rolling_mean_168       168
high_demand_flag         0
extreme_demand_flag      0
temp_mean                0
temp_max                 0
temp_min                 0
humid_mean               0
dewpoint_mean            0
cloud_cover_mean         0
apparent_temp_mean       0
wind_speed_mean          0
precip_mean              0
is_weekend               0
is_summer                0
is_winter                0
dtype: int64

Validated engineered features.

In [9]:
df_model_clean = df_model.dropna(
    subset = [
        'lag_1',
        'lag_24',
        'lag_168',
        'rolling_mean_24',
        'rolling_mean_168'
    ]
).copy()

df_model_clean.head()

,period_utc,value,hour,dayofweek,month,year,lag_1,lag_24,lag_168,rolling_mean_24,...,temp_min,humid_mean,dewpoint_mean,cloud_cover_mean,apparent_temp_mean,wind_speed_mean,precip_mean,is_weekend,is_summer,is_winter
168,2019-01-08 00:00:00-06:00,32224.0,6,1,1,2019,34538.0,31505.0,37301.0,34802.333333,...,53.690002,78.130089,51.213047,74.500000,57.522842,7.279257,0.0,False,False,True
169,2019-01-08 01:00:00-06:00,30574.0,7,1,1,2019,32224.0,30029.0,36953.0,34832.291667,...,52.160000,79.633080,51.198048,84.833336,56.926517,6.678637,0.0,False,False,True
170,2019-01-08 02:00:00-06:00,29701.0,8,1,1,2019,30574.0,28963.0,37114.0,34855.000000,...,49.369999,81.034370,51.183048,88.000000,56.392902,5.894207,0.0,False,False,True
171,2019-01-08 03:00:00-06:00,29397.0,9,1,1,2019,29701.0,28581.0,37154.0,34885.750000,...,47.119999,82.420723,50.313049,95.666664,54.640015,5.928419,0.0,False,False,True
172,2019-01-08 04:00:00-06:00,29496.0,10,1,1,2019,29397.0,28518.0,37290.0,34919.750000,...,45.320000,83.084534,50.298050,91.500000,54.643185,5.114326,0.0,False,False,True


In [10]:
df_model_clean.shape

(61177, 25)

Handling null values in lag and rolling mean features. These require previous values to compute and a week is a comparatively low amount compared to the whole dataset. Imputation would add unnecessary bias.

Baseline model

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

num_features = [
    'hour',
    'dayofweek',
    'month',
    'year',
    'lag_1',
    'lag_24',
    'lag_168',
    'rolling_mean_24',
    'rolling_mean_168',
    'temp_mean',
    'temp_max',
    'temp_min',
    'humid_mean',
    'dewpoint_mean',
    'cloud_cover_mean',
    'apparent_temp_mean',
    'wind_speed_mean',
    'precip_mean',
    'is_weekend',
    'is_summer',
    'is_winter'
]

cat_features = []

num_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_preprocessor, num_features),
    ('cat', cat_preprocessor, cat_features)
])

def make_model_pipeline(model):
        return Pipeline(steps =[
            ('preprocessor', preprocessor),
            ('model', model)
    ])

target = 'value'

feature_cols = num_features + cat_features

X=df_model_clean[feature_cols]
y=df_model_clean[target]

train_mask = df_model_clean['period_utc'] < '2025-01-01'
test_mask =df_model_clean['period_utc'] >= '2025-01-01'

X_train = X.iloc[train_mask]
X_test = X.iloc[test_mask]

y_train = y.iloc[train_mask]
y_test = y.iloc[test_mask]

In [12]:
splits = pd.DataFrame({
    'splits': ['train', 'test'],
    'rows': [train_mask.sum(), test_mask.sum()],
    'start': [
        df_model_clean.loc[X_train.index, 'period_utc'].min(),
        df_model_clean.loc[X_test.index, 'period_utc'].min()
    ],
    'end': [
        df_model_clean.loc[X_train.index, 'period_utc'].max(),
        df_model_clean.loc[X_test.index, 'period_utc'].max()
    ]
})

splits

,splits,rows,start,end
0,train,52440,2019-01-08 00:00:00-06:00,2024-12-31 23:00:00-06:00
1,test,8737,2025-01-01 00:00:00-06:00,2025-12-31 00:00:00-06:00


In [13]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

base_grid = {
    'model__alpha': [.01,.1,1,10,100]
}

base_search = GridSearchCV(
    estimator= make_model_pipeline(Ridge()),
    param_grid = base_grid,
    scoring = 'neg_mean_absolute_error',
    cv=tscv,
    n_jobs=-1
)
base_search.fit(X_train, y_train)
base_params, base_best_score = base_search.best_params_, -(base_search.best_score_)
best_base = base_search.best_estimator_
base_pred_train = best_base.predict(X_train)
base_pred_test = best_base.predict(X_test)

base_mae_train = mean_absolute_error(y_train, base_pred_train)
base_mae_test = mean_absolute_error(y_test, base_pred_test)
base_rmse_test = root_mean_squared_error(y_test, base_pred_test)
base_rmse_train = root_mean_squared_error(y_train, base_pred_train)
base_gap_mae = base_mae_test - base_mae_train
base_gap_rmse = base_rmse_test - base_rmse_train
print('Best model parameters are', base_params, f'Baseline MAE is: {base_mae_test:.2f}. Baseline RMSE is: {base_rmse_test:.2f}.')

Best model parameters are {'model__alpha': 1} Baseline MAE is: 1113.06. Baseline RMSE is: 1405.77.


In [14]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

rf_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 5, 10],
    'model__max_features': ['sqrt', .5, 1],

}
rf_search = GridSearchCV(
    estimator = make_model_pipeline(RandomForestRegressor()),
    param_grid = rf_grid,
    scoring = 'neg_mean_absolute_error',
    cv = tscv,
    n_jobs = -1
)

rf_search.fit(X_train, y_train)
rf_params, rf_best_score = rf_search.best_params_, -(rf_search.best_score_)
best_rf = rf_search.best_estimator_
rf_pred_train = best_rf.predict(X_train)
rf_pred_test = best_rf.predict(X_test)

rf_mae_train = mean_absolute_error(y_train, rf_pred_train)
rf_mae_test = mean_absolute_error(y_test, rf_pred_test)
rf_rmse_train = root_mean_squared_error(y_train, rf_pred_train)
rf_rmse_test = root_mean_squared_error(y_test, rf_pred_test)
rf_gap_mae = rf_mae_test - rf_mae_train
rf_gap_rmse = rf_rmse_test - rf_rmse_train
print('Best RF model parameters are', rf_params, f'RF MAE is: {rf_mae_test:.2f}. RF RMSE is: {rf_rmse_test:.2f}.')

Best RF model parameters are {'model__max_depth': 15, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100} RF MAE is: 654.01. RF RMSE is: 893.38.


In [15]:
hgb_grid ={
    'model__max_iter': [100, 300, 500],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__max_leaf_nodes': [15, 31, 63],
    'model__min_samples_leaf': [20, 50, 100],
    'model__l2_regularization': [0.0, 0.1, 1.0]
}

hgb_search = GridSearchCV(
    estimator= make_model_pipeline(HistGradientBoostingRegressor()),
    param_grid = hgb_grid,
    scoring = 'neg_mean_absolute_error',
    cv = tscv,
    n_jobs = -1
)

hgb_search.fit(X_train, y_train)
hgb_params, hgb_best_score = hgb_search.best_params_, hgb_search.best_score_
best_hgb = hgb_search.best_estimator_

hgb_pred_train = best_hgb.predict(X_train)
hgb_pred_test = best_hgb.predict(X_test)

hgb_mae_train = mean_absolute_error(y_train, hgb_pred_train)
hgb_mae_test = mean_absolute_error(y_test, hgb_pred_test)
hgb_rmse_train = root_mean_squared_error(y_train, hgb_pred_train)
hgb_rmse_test = root_mean_squared_error(y_test, hgb_pred_test)
hgb_gap_mae = hgb_mae_test - hgb_mae_train
hgb_gap_rmse = hgb_rmse_test - hgb_rmse_train
print('Best HGB model parameters are', hgb_params, f'HGB MAE is: {hgb_mae_test:.2f}. HGB RMSE is: {hgb_rmse_test:.2f}.')

Best HGB model parameters are {'model__l2_regularization': 0.1, 'model__learning_rate': 0.05, 'model__max_iter': 500, 'model__max_leaf_nodes': 63, 'model__min_samples_leaf': 20} HGB MAE is: 469.58. HGB RMSE is: 625.16.


In [16]:
models = ['Ridge', 'RF', 'HGB']
mae_train = [base_mae_train, rf_mae_train, hgb_mae_train]
mae_test = [base_mae_test, rf_mae_test, hgb_mae_test]
rmse_train = [base_rmse_train, rf_rmse_train, hgb_rmse_train]
rmse_test = [base_rmse_test, rf_rmse_test, hgb_rmse_test]
mae_gap = [base_gap_mae, rf_gap_mae, hgb_gap_mae]
rmse_gap = [base_gap_rmse, rf_gap_rmse, hgb_gap_rmse]

def final_results_table(models, mae_train, mae_test, mae_gap, rmse_train, rmse_test, rmse_gap):
    rows = []
    for model, train_mae, test_mae, gap_mae, train_rmse, test_rmse,  gap_rmse in zip(
            models, mae_train, mae_test, mae_gap, rmse_train, rmse_test,  rmse_gap):
        rows.append({
             'model':model,
            'mae_train': train_mae,
            'mae_test': test_mae,
            'gap_mae': gap_mae,
            'rmse_train': train_rmse,
            'rmse_test': test_rmse,
            'gap_rmse': gap_rmse
        })
    return pd.DataFrame(rows)

final_results_table(models, mae_train, mae_test, mae_gap, rmse_train, rmse_test, rmse_gap)

,model,mae_train,mae_test,gap_mae,rmse_train,rmse_test,gap_rmse
0,Ridge,1073.218926,1113.056303,39.837378,1365.103947,1405.765555,40.661608
1,RF,229.481470,654.008228,424.526757,303.532724,893.378483,589.845760
2,HGB,227.701005,469.576803,241.875798,302.097209,625.158920,323.061711


In [17]:
percentage_table = pd.DataFrame({
    'model': models,
    'mae_percentage': pd.Series(mae_test) / y_test.mean(),
    'rmse_percentage': pd.Series(rmse_test) / y_test.mean()
}
)

percentage_table.style.format({
    "mae_percentage": "{:.2%}",
    "rmse_percentage": "{:.2%}"
})

,model,mae_percentage,rmse_percentage
0,Ridge,2.00%,2.52%
1,RF,1.17%,1.60%
2,HGB,0.84%,1.12%


Mean Average Error is the primary selection metric as it represents the average forecast error in MW with Root Mean Squared Error serving as a diagnostic. RMSE assists with accounting for large misses in forecasting at peak demand which is operationally critical.  Using percentages, we can see that the lowest performing model is only off by 2% on average with the preferred model averaging an error percentage of .85 across the model.

HGB shows the lowest amount of testing error in both MAE and RMSE when testing our models. Our biggest concern when forecasting is how the models handle predicting when in the upper demand bins. Going to filter to two upper demand percentiles and compare metrics across.

In [18]:
model_results = pd.DataFrame({
    'period_utc': df_model_clean.loc[X_test.index]['period_utc'].values,
    'y_true': y_test.values,
    'ridge_pred': base_pred_test,
    'rf_pred': rf_pred_test,
    'hgb_pred': hgb_pred_test

})
model_results

,period_utc,y_true,ridge_pred,rf_pred,hgb_pred
0,2025-01-01 06:00:00,43875.0,44206.410791,43521.215927,43142.644701
1,2025-01-01 07:00:00,43161.0,43243.549901,42394.101707,42417.496741
2,2025-01-01 08:00:00,43085.0,42532.485594,42097.876028,42323.933025
3,2025-01-01 09:00:00,42870.0,42417.440121,42318.504392,42895.799822
4,2025-01-01 10:00:00,42709.0,42293.811541,43119.638751,43513.479608
...,...,...,...,...,...
8732,2025-12-31 02:00:00,55192.0,54841.965607,53938.121960,54949.710603
8733,2025-12-31 03:00:00,55664.0,55307.388312,53958.831515,55028.959364
8734,2025-12-31 04:00:00,55464.0,55567.690797,53767.205722,54456.016982
8735,2025-12-31 05:00:00,54362.0,55099.037292,53514.930809,53637.950843


In [19]:
hi_demand_threshold = y_train.quantile(.95)
ex_demand_threshold = y_train.quantile(.99)

masks = {
    'normal': model_results['y_true'] < hi_demand_threshold,
    'high_demand': model_results['y_true'] >= hi_demand_threshold,
    'extreme_demand': model_results['y_true'] >= ex_demand_threshold
}
masks.items()

dict_items([('normal', 0       True
1       True
2       True
3       True
4       True
        ... 
8732    True
8733    True
8734    True
8735    True
8736    True
Name: y_true, Length: 8737, dtype: bool), ('high_demand', 0       False
1       False
2       False
3       False
4       False
        ...  
8732    False
8733    False
8734    False
8735    False
8736    False
Name: y_true, Length: 8737, dtype: bool), ('extreme_demand', 0       False
1       False
2       False
3       False
4       False
        ...  
8732    False
8733    False
8734    False
8735    False
8736    False
Name: y_true, Length: 8737, dtype: bool)])

In [34]:
pred_cols = ['ridge_pred', 'rf_pred', 'hgb_pred']
def eval_segment(df, cols, masks):
    rows = []
    for segment_name, mask in masks.items():
        for col in cols:
            mae = mean_absolute_error(
                    df.loc[mask, 'y_true'],
                    df.loc[mask, col]
            )
            rmse = root_mean_squared_error(
                df.loc[mask, 'y_true'],
                df.loc[mask, col]
            )
            mae_percentage = mae / df.loc[mask,'y_true'].mean()
            rmse_percentage = rmse / df.loc[mask, 'y_true'].mean()
            rows.append({
                'segment': segment_name,
                'model': col,
                'mae':  mae,
                'mae_percentage': mae_percentage,
                'rmse': rmse,
                'rmse_percentage': rmse_percentage
            })
    return rows

eval_results_demand = pd.DataFrame(eval_segment(model_results, pred_cols, masks)).sort_values(['segment','mae'], ascending=[True, True])
eval_results_demand.style.format({
    'mae': '{:,.2f}',
    'mae_percentage': '{:.2%}',
    'rmse': '{:,.2f}',
    'rmse_percentage': '{:.2%}'
    })

,segment,model,mae,mae_percentage,rmse,rmse_percentage
6,extreme_demand,ridge_pred,838.49,1.04%,"1,099.17",1.36%
8,extreme_demand,hgb_pred,859.21,1.07%,"1,083.87",1.34%
7,extreme_demand,rf_pred,876.14,1.09%,"1,115.33",1.38%
5,high_demand,hgb_pred,710.61,0.95%,876.10,1.17%
4,high_demand,rf_pred,965.33,1.29%,"1,274.87",1.70%
3,high_demand,ridge_pred,"1,002.66",1.34%,"1,285.44",1.72%
2,normal,hgb_pred,432.93,0.82%,577.54,1.09%
1,normal,rf_pred,606.68,1.15%,819.98,1.55%
0,normal,ridge_pred,"1,129.84",2.14%,"1,423.17",2.69%


Analyzing the results from the different demand segments, the lowest error in all segments is still HGB. The model still has relatively low error percentages in all segments.

In [35]:
model_results['temp_mean'] = df_model_clean.loc[y_test.index, 'temp_mean'].values
model_results.head()

,period_utc,y_true,ridge_pred,rf_pred,hgb_pred,temp_mean
0,2025-01-01 06:00:00,43875.0,44206.410791,43521.215927,43142.644701,49.383053
1,2025-01-01 07:00:00,43161.0,43243.549901,42394.101707,42417.496741,48.288052
2,2025-01-01 08:00:00,43085.0,42532.485594,42097.876028,42323.933025,47.268051
3,2025-01-01 09:00:00,42870.0,42417.440121,42318.504392,42895.799822,46.383053
4,2025-01-01 10:00:00,42709.0,42293.811541,43119.638751,43513.479608,45.738052


In [36]:
train_temps = df_model_clean.loc[y_train.index, 'temp_mean']

temp_masks = {
    'hot_mask': model_results['temp_mean'] >= train_temps.quantile(.95),
    'normal_mask': model_results['temp_mean'].between(
        train_temps.quantile(.05),
        train_temps.quantile(.95)
    ),
    'cold_mask': model_results['temp_mean'] <= train_temps.quantile(.05)

}
temp_masks.keys()

dict_keys(['hot_mask', 'normal_mask', 'cold_mask'])

In [38]:
eval_results_temp = pd.DataFrame(eval_segment(model_results, pred_cols, temp_masks))
eval_results_temp.sort_values(['segment', 'mae'], ascending=[True, True], inplace=True)

eval_results_temp = pd.DataFrame(eval_segment(model_results, pred_cols, masks)).sort_values(['segment','mae'], ascending=[True, True])
eval_results_temp.style.format({
    'mae': '{:,.2f}',
    'mae_percentage': '{:.2%}',
    'rmse': '{:,.2f}',
    'rmse_percentage': '{:.2%}'
    })

,segment,model,mae,mae_percentage,rmse,rmse_percentage
6,extreme_demand,ridge_pred,838.49,1.04%,"1,099.17",1.36%
8,extreme_demand,hgb_pred,859.21,1.07%,"1,083.87",1.34%
7,extreme_demand,rf_pred,876.14,1.09%,"1,115.33",1.38%
5,high_demand,hgb_pred,710.61,0.95%,876.10,1.17%
4,high_demand,rf_pred,965.33,1.29%,"1,274.87",1.70%
3,high_demand,ridge_pred,"1,002.66",1.34%,"1,285.44",1.72%
2,normal,hgb_pred,432.93,0.82%,577.54,1.09%
1,normal,rf_pred,606.68,1.15%,819.98,1.55%
0,normal,ridge_pred,"1,129.84",2.14%,"1,423.17",2.69%


Analyzing the temperature-band results continues to show that HGB has the lowest error in every segment. Error percentages are still low.

In the end, ERCOT power demand is highly autocorrelated. Lag features provides strong baseline predictive power with weather adding context, especially in the case of the non-linear response of demand to temperature extremes. HistGradientBoosting performed best indicating that non-linear models are better suited for demand behavior.